# Pipeline audit metrics

This notebook loads the archived result artifacts used by `README.md`. It deliberately does not rerun model inference. Every displayed value is measured unless a note explicitly says otherwise. Run it from the repository root.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'notebooks/archive').exists():
    ROOT = Path('../..').resolve()
ARCHIVE = ROOT / 'notebooks/archive'
CURRENT = ARCHIVE / 'results/current'
LVLM = ARCHIVE / 'results/lvlm'
assert CURRENT.exists() and LVLM.exists()

## Action-generation prompt lineage (90 human-labeled audit-v11 clips)

In [2]:
versions = {
    'base': 'qwen3.5_audit_v11_seg_extract_spec_eval.spec_f1_overall_aggregate.csv',
    'CVS context v1': 'qwen3.5_audit_v11_seg_extract_spec_eval_with_cvs_context.spec_f1_overall_aggregate.csv',
    'CVS context v2': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v2__baseline.spec_f1_overall_aggregate.csv',
    'CVS context v3': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3.spec_f1_overall_aggregate.csv',
    'CVS context v4': 'qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v4.spec_f1_overall_aggregate.csv',
}
rows = []
for version, filename in versions.items():
    frame = pd.read_csv(CURRENT / filename)
    score_col = 'aggregate mean F1' if 'aggregate mean F1' in frame else 'f1_mean_avg'
    count_col = 'rows' if 'rows' in frame else 'n_rows'
    for _, row in frame.iterrows():
        rows.append({'version': version, 'method': row['method'], 'spec_f1': row[score_col], 'summary_rows': row[count_col], 'source': filename})
lineage = pd.DataFrame(rows)
lineage.pivot(index='version', columns='method', values='spec_f1').round(3)

method,description_only,description_only_cvs_context,hint_questions,hint_questions_cvs_context,structured_prediction,structured_prediction_cvs_context,structured_prediction_deterministic,structured_prediction_deterministic_cvs_context
version,,,,,,,,
CVS context v1,NaN,0.185,NaN,0.262,NaN,0.302,NaN,0.271
CVS context v2,NaN,0.215,NaN,0.269,NaN,0.326,NaN,0.275
CVS context v3,NaN,0.199,NaN,0.282,NaN,0.323,NaN,0.276
CVS context v4,NaN,0.186,NaN,0.299,NaN,0.340,NaN,0.263
base,0.165,NaN,0.156,NaN,0.195,NaN,0.158,NaN


In [3]:
actor_rows = []
for version, filename in versions.items():
    actor_filename = filename.replace('spec_f1_overall_aggregate', 'spec_f1_actor_aggregate_table')
    frame = pd.read_csv(CURRENT / actor_filename)
    if 'method' in frame.columns:
        method_name = 'structured_prediction' if version == 'base' else 'structured_prediction_cvs_context'
        frame = frame[frame['method'] == method_name]
        for _, row in frame.iterrows():
            actor_rows.append({'version': version, 'actor': row['actor'], 'structured_actor_f1': row['f1_mean_avg']})
    else:
        for _, row in frame.iterrows():
            actor_rows.append({'version': version, 'actor': row['actor'], 'structured_actor_f1': row['structured_prediction_cvs_context']})
pd.DataFrame(actor_rows).pivot(index='version', columns='actor', values='structured_actor_f1').round(3)

actor,camera,left,other,right
version,,,,
CVS context v1,0.169,0.514,0.333,0.213
CVS context v2,0.182,0.576,0.417,0.189
CVS context v3,0.168,0.588,0.333,0.211
CVS context v4,0.171,0.572,0.667,0.168
base,0.086,0.322,0.333,0.131


v3.1 and v3.2 are post-processing variants of v3. The next cells recompute their structured-prediction aggregate F1 from the archived evaluation rows and count the changes recorded by their override logs.

In [4]:
postprocess_scores = []
for version in ('v3_1', 'v3_2'):
    data = json.loads((CURRENT / f'{version}_spec_eval_results.json').read_text())
    rows = data['structured_prediction_cvs_context']
    right_rows = [row for row in rows if row['actor'] == 'right']
    postprocess_scores.append({
        'version': version.replace('_', '.'),
        'structured_spec_f1': sum(row['f1_mean'] for row in rows) / len(rows),
        'structured_right_actor_f1': sum(row['f1_mean'] for row in right_rows) / len(right_rows),
        'summary_rows': len(rows),
    })
pd.DataFrame(postprocess_scores).round(6)

,version,structured_spec_f1,structured_right_actor_f1,summary_rows
0,v3.1,0.329713,0.232169,30
1,v3.2,0.329243,0.230601,30


In [5]:
override_rows = []
for version in ('v3_1', 'v3_2'):
    data = json.loads((CURRENT / f'{version}_override_summary.json').read_text())
    override_rows.append({
        'version': version.replace('_', '.'),
        'logged_rows': len(data),
        'examples': len({row['example_id'] for row in data}),
        'overrides_applied': sum(bool(row['override_applied']) for row in data),
    })
pd.DataFrame(override_rows)

,version,logged_rows,examples,overrides_applied
0,v3.1,360,90,39
1,v3.2,561,89,70


In [6]:
full_override_rows = []
for split in ('train_full', 'test_full'):
    data = json.loads((CURRENT / f'v3_1_{split}_override_summary.json').read_text())
    full_override_rows.append({
        'split': split,
        'logged_rows': len(data),
        'examples': len({row['example_id'] for row in data}),
        'rubric_predictions_present': sum(bool(row.get('rubric_clip_tool')) for row in data),
        'overrides_applied': sum(bool(row['override_applied']) for row in data),
    })
pd.DataFrame(full_override_rows)

,split,logged_rows,examples,rubric_predictions_present,overrides_applied
0,train_full,1314,1312,0,0
1,test_full,481,481,90,0


## Selected v3.1 held-out results

In [7]:
scores = pd.read_csv(CURRENT / 'validation_test_final_scores_seed20260610.csv')
selected = scores[scores.method_id == 'structured_llm_extract_tool_fix'][
    ['split', 'left_exact', 'right_medium', 'camera_coarse', 'final_score']
].copy()
selected.round(3)

,split,left_exact,right_medium,camera_coarse,final_score
8,validation,0.644,0.295,0.346,0.428
9,test,0.461,0.308,0.194,0.321


In [8]:
corr = pd.read_csv(CURRENT / 'recomputed_six_system_human_synthetic_final_score_correlation_test20.csv')
corr[['n_systems', 'n_test_videos_in_seed_split', 'pearson_r', 'pearson_bootstrap_ci_low', 'pearson_bootstrap_ci_high', 'spearman_r', 'spearman_bootstrap_ci_low', 'spearman_bootstrap_ci_high']].round(3)

,n_systems,n_test_videos_in_seed_split,pearson_r,pearson_bootstrap_ci_low,pearson_bootstrap_ci_high,spearman_r,spearman_bootstrap_ci_low,spearman_bootstrap_ci_high
0,6,20,0.947,0.27,0.968,0.886,0.2,1.0


The direct scores and the correlation answer different questions: direct scores estimate agreement with human action labels; the six-point correlation estimates whether synthetic GT ranks six downstream systems similarly.

## Additional right-tool checker

In [9]:
right_tool_nb = json.loads((ARCHIVE / 'experimental_action_pipeline/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps.ipynb').read_text())
all_output_text = '\n'.join(
    ''.join(output.get('text', [])) + ''.join(output.get('data', {}).get('text/plain', []))
    for cell in right_tool_nb['cells'] for output in cell.get('outputs', [])
)
assert '0.568182' in all_output_text and '0.590909' in all_output_text
pd.DataFrame([
    {'method': 'v3 structured clip tool', 'n': 88, 'accuracy': 0.590909},
    {'method': 'rubric + 10 shape examples', 'n': 88, 'accuracy': 0.568182},
]).round(3)

,method,n,accuracy
0,v3 structured clip tool,88,0.591
1,rubric + 10 shape examples,88,0.568


## Earlier `lvlm_cvs_reasoning` ablations

In [10]:
lvlm_runs = {
    'single / direct / no organs': 'vanilla_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'sequential / direct / no organs': 'sequential_inductive_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'single / direct / organs': 'vanilla_organs_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'sequential / direct / organs': 'sequential_inductive_organs_gpt-4.1-mini_train_dev5_frames_pred.metrics.json',
    'single / observe-then-decide / no organs': 'vanilla_gpt-4.1-mini_train_dev5_frames_obs_then_decide.metrics.json',
    'sequential / observe-then-decide / no organs': 'sequential_inductive_gpt-4.1-mini_train_dev5_frames_obs_then_decide.metrics.json',
}
lvlm_rows = []
for label, filename in lvlm_runs.items():
    row = json.loads((LVLM / filename).read_text())
    lvlm_rows.append({'configuration': label, **row, 'source': filename})
lvlm = pd.DataFrame(lvlm_rows)
lvlm[['configuration', 'num_frames', 'AP_c1', 'AP_c2', 'AP_c3', 'mAP_criteria']].round(3)

,configuration,num_frames,AP_c1,AP_c2,AP_c3,mAP_criteria
0,single / direct / no organs,90,0.258,0.431,0.222,0.304
1,sequential / direct / no organs,90,0.368,0.484,0.334,0.395
2,single / direct / organs,90,0.249,0.450,0.339,0.346
3,sequential / direct / organs,90,0.233,0.452,0.261,0.316
4,single / observe-then-decide / no organs,90,0.311,0.425,0.222,0.319
5,sequential / observe-then-decide / no organs,90,0.366,0.528,0.291,0.395


In [11]:
organs = pd.read_csv(LVLM / 'organ_sweep_summary.csv')
organs[['organ_variant', 'num_frames', 'AP_c1', 'AP_c2', 'AP_c3', 'mAP_criteria']].sort_values('mAP_criteria', ascending=False).round(3)

,organ_variant,num_frames,AP_c1,AP_c2,AP_c3,mAP_criteria
1,orgseg_overlay_contour,90,0.357,0.478,0.306,0.381
2,orgseg_overlay_fill,90,0.348,0.437,0.268,0.351
0,orgbbox_text_bbox_topk,90,0.259,0.434,0.264,0.319
3,orgseg_text_centroid_topk,90,0.253,0.427,0.235,0.305
4,orgseg_text_stats_topk,90,0.254,0.408,0.232,0.298


These earlier results are a separate CVS-achievement task, not action-segmentation accuracy. They influenced the design choice to include temporal/CVS context, but their mAP values must not be presented as v3.1 action-label performance. The experiments use only 90 frames from five development videos, so qualitative claims about temporal prompting or organ overlays are exploratory, not confirmed on an independent test set.